# Advanced Problems with Solutions: In-Place Concatenation and Repetition

Topic: Python sequence operations involving `+`, `+=`, `*`, and `*=`

This notebook contains advanced practice problems with complete solutions.

Main ideas covered:

- Difference between concatenation and in-place concatenation
- Difference between repetition and in-place repetition
- Mutable versus immutable sequence behavior
- Object identity with `id`
- Aliasing and side effects
- List `+=` versus `list.extend`
- `append` versus `extend` versus `+=`
- Tuple fallback behavior
- Strings and immutable augmented assignment
- Nested mutable containers
- Function side effects
- Mutable default arguments
- Generator consumption
- Custom classes with `__add__`, `__iadd__`, `__mul__`, and `__imul__`
- Debugging production-style bugs caused by in-place mutation

Best practice: run each problem cell first, make a prediction, then run the solution cell.


## Setup Helpers

These helpers are used in several problems to make object identity and aliasing easier to inspect.


In [1]:
def show_identity(name, obj):
    print(f"{name}: id={id(obj)}, value={obj!r}")


def same_object(label, a, b):
    print(f"{label}: {a is b}")


def separator(title=None):
    if title:
        print("\n" + "=" * 20, title, "=" * 20)
    else:
        print("\n" + "=" * 50)


## Problem 1 — List `+=` Mutates the Existing Object


Predict the output.

Pay attention to:

- the value of `a`
- the value of `alias`
- whether the object identity changes
- whether both names still refer to the same object


In [2]:
a = [1, 2]
alias = a
extra = [3, 4]

before = id(a)
a += extra
after = id(a)

print(a)
print(alias)
print(before == after)
print(a is alias)


[1, 2, 3, 4]
[1, 2, 3, 4]
True
True


### Solution 1


Expected output:

```text
[1, 2, 3, 4]
[1, 2, 3, 4]
True
True
```

For lists, `+=` performs an in-place extension. The original list object is mutated.

Since `alias` points to the same list object as `a`, it observes the mutation too.


In [3]:
a = [1, 2]
alias = a
extra = [3, 4]

before = id(a)
a += extra
after = id(a)

assert a == [1, 2, 3, 4]
assert alias == [1, 2, 3, 4]
assert before == after
assert a is alias

print("All assertions passed.")


All assertions passed.


## Problem 2 — List `+` Creates a New Object


Predict the output.

This problem contrasts `a = a + extra` with `a += extra`.


In [4]:
a = [1, 2]
alias = a
extra = [3, 4]

before = id(a)
a = a + extra
after = id(a)

print(a)
print(alias)
print(before == after)
print(a is alias)


[1, 2, 3, 4]
[1, 2]
False
False


### Solution 2


Expected output:

```text
[1, 2, 3, 4]
[1, 2]
False
False
```

`a + extra` creates a new list. Then the name `a` is rebound to that new list.

The old list is still referenced by `alias`, so `alias` remains unchanged.


In [5]:
a = [1, 2]
alias = a
extra = [3, 4]

before = id(a)
a = a + extra
after = id(a)

assert a == [1, 2, 3, 4]
assert alias == [1, 2]
assert before != after
assert a is not alias

print("All assertions passed.")


All assertions passed.


## Problem 3 — `+=` Accepts Any Iterable for Lists


For lists, `+=` behaves similarly to `extend`.

Predict the final value of `items`.


In [6]:
items = [10]
items += (20, 30)
items += range(40, 43)
items += "ab"

print(items)


[10, 20, 30, 40, 41, 42, 'a', 'b']


### Solution 3


Expected output:

```text
[10, 20, 30, 40, 41, 42, 'a', 'b']
```

A list can be extended by any iterable.

The tuple contributes `20` and `30`.

The range contributes `40`, `41`, and `42`.

The string contributes its individual characters, `'a'` and `'b'`.


In [7]:
items = [10]
items += (20, 30)
items += range(40, 43)
items += "ab"

assert items == [10, 20, 30, 40, 41, 42, "a", "b"]
print("All assertions passed.")


All assertions passed.


## Problem 4 — `append` versus `+=`


Fill in the correct method to produce each target.

Starting value:

```python
lst = [1, 2]
```

Targets:

```python
[1, 2, [3, 4]]
[1, 2, 3, 4]
[1, 2, 'ab']
[1, 2, 'a', 'b']
```


In [8]:
# Try your own answers first.

lst = [1, 2]
# produce [1, 2, [3, 4]]

lst = [1, 2]
# produce [1, 2, 3, 4]

lst = [1, 2]
# produce [1, 2, 'ab']

lst = [1, 2]
# produce [1, 2, 'a', 'b']


### Solution 4


Solutions:

```python
lst = [1, 2]
lst.append([3, 4])
```

```python
lst = [1, 2]
lst += [3, 4]
```

```python
lst = [1, 2]
lst.append("ab")
```

```python
lst = [1, 2]
lst += "ab"
```

`append` adds one object as a single element.

`+=` extends the list by iterating over the right-hand operand.


In [9]:
lst = [1, 2]
lst.append([3, 4])
assert lst == [1, 2, [3, 4]]

lst = [1, 2]
lst += [3, 4]
assert lst == [1, 2, 3, 4]

lst = [1, 2]
lst.append("ab")
assert lst == [1, 2, "ab"]

lst = [1, 2]
lst += "ab"
assert lst == [1, 2, "a", "b"]

print("All assertions passed.")


All assertions passed.


## Problem 5 — Tuple `+=` Creates a New Tuple


Predict the output.

Tuples are immutable, so `+=` cannot mutate them in place.


In [10]:
t = (1, 2)
alias = t

before = id(t)
t += (3, 4)
after = id(t)

print(t)
print(alias)
print(before == after)
print(t is alias)


(1, 2, 3, 4)
(1, 2)
False
False


### Solution 5


Expected output:

```text
(1, 2, 3, 4)
(1, 2)
False
False
```

For tuples, `t += (3, 4)` behaves like:

```python
t = t + (3, 4)
```

A new tuple is created and assigned back to `t`.

The name `alias` still points to the original tuple.


In [11]:
t = (1, 2)
alias = t

before = id(t)
t += (3, 4)
after = id(t)

assert t == (1, 2, 3, 4)
assert alias == (1, 2)
assert before != after
assert t is not alias

print("All assertions passed.")


All assertions passed.


## Problem 6 — String `+=` Rebinds the Name


Predict the output.

Strings are immutable, like tuples.


In [12]:
s = "py"
alias = s

before = id(s)
s += "thon"
after = id(s)

print(s)
print(alias)
print(before == after)
print(s is alias)


python
py
False
False


### Solution 6


Expected output:

```text
python
py
False
False
```

Strings cannot be mutated.

So `s += "thon"` creates a new string and rebinds `s`.

The name `alias` still points to the original string `"py"`.


In [13]:
s = "py"
alias = s

before = id(s)
s += "thon"
after = id(s)

assert s == "python"
assert alias == "py"
assert s is not alias

print("All assertions passed.")


All assertions passed.


## Problem 7 — In-Place Repetition with Lists


Predict the output.

For lists, `*=` mutates the list object in place.


In [14]:
nums = [1, 2]
alias = nums

before = id(nums)
nums *= 3
after = id(nums)

print(nums)
print(alias)
print(before == after)
print(nums is alias)


[1, 2, 1, 2, 1, 2]
[1, 2, 1, 2, 1, 2]
True
True


### Solution 7


Expected output:

```text
[1, 2, 1, 2, 1, 2]
[1, 2, 1, 2, 1, 2]
True
True
```

For a list, `*=` performs in-place repetition.

The list object is mutated, so aliases see the updated contents.


In [15]:
nums = [1, 2]
alias = nums

before = id(nums)
nums *= 3
after = id(nums)

assert nums == [1, 2, 1, 2, 1, 2]
assert alias == nums
assert before == after
assert nums is alias

print("All assertions passed.")


All assertions passed.


## Problem 8 — Repetition with Tuples


Predict the output.

For tuples, `*=` creates a new tuple.


In [16]:
t = (1, 2)
alias = t

before = id(t)
t *= 3
after = id(t)

print(t)
print(alias)
print(before == after)
print(t is alias)


(1, 2, 1, 2, 1, 2)
(1, 2)
False
False


### Solution 8


Expected output:

```text
(1, 2, 1, 2, 1, 2)
(1, 2)
False
False
```

Tuples are immutable.

`t *= 3` behaves like:

```python
t = t * 3
```

The name `t` is rebound to a new tuple.


In [17]:
t = (1, 2)
alias = t

before = id(t)
t *= 3
after = id(t)

assert t == (1, 2, 1, 2, 1, 2)
assert alias == (1, 2)
assert t is not alias
assert before != after

print("All assertions passed.")


All assertions passed.


## Problem 9 — Nested Lists and Repeated References


Predict the output.

This is one of the most common bugs involving repetition.


In [18]:
matrix = [[0]]
matrix *= 3

matrix[0].append(99)

print(matrix)
print(matrix[0] is matrix[1])
print(matrix[1] is matrix[2])


[[0, 99], [0, 99], [0, 99]]
True
True


### Solution 9


Expected output:

```text
[[0, 99], [0, 99], [0, 99]]
True
True
```

`matrix *= 3` repeats references to the same inner list.

It does not create three independent inner lists.

So mutating one row mutates the shared inner list visible from every position.


In [19]:
matrix = [[0]]
matrix *= 3

matrix[0].append(99)

assert matrix == [[0, 99], [0, 99], [0, 99]]
assert matrix[0] is matrix[1]
assert matrix[1] is matrix[2]

print("All assertions passed.")


All assertions passed.


## Problem 10 — Correctly Creating Independent Rows


Fix the following code so that each row is independent.

Buggy version:

```python
grid = [[0] * 3] * 4
grid[0][0] = 99
```

Expected result:

```python
[[99, 0, 0], [0, 0, 0], [0, 0, 0], [0, 0, 0]]
```


In [20]:
# Write the corrected version here.


### Solution 10


Use a list comprehension:

```python
grid = [[0] * 3 for _ in range(4)]
grid[0][0] = 99
```

The inner expression `[0] * 3` is evaluated once per loop iteration.

That means each row is a distinct list.


In [21]:
grid = [[0] * 3 for _ in range(4)]
grid[0][0] = 99

assert grid == [[99, 0, 0], [0, 0, 0], [0, 0, 0], [0, 0, 0]]
assert grid[0] is not grid[1]
assert grid[1] is not grid[2]
assert grid[2] is not grid[3]

print(grid)
print("All assertions passed.")


[[99, 0, 0], [0, 0, 0], [0, 0, 0], [0, 0, 0]]
All assertions passed.


## Problem 11 — The Tuple-Containing-a-List Trap


Predict the output.

This is a subtle augmented assignment behavior.


In [22]:
t = ([1, 2], [3, 4])

try:
    t[0] += [99]
except TypeError as ex:
    print(type(ex).__name__)

print(t)


TypeError
([1, 2, 99], [3, 4])


### Solution 11


Expected output:

```text
TypeError
([1, 2, 99], [3, 4])
```

This happens because augmented assignment has multiple stages.

First, Python retrieves `t[0]`, which is a list.

Then it performs in-place addition on that list. The list is mutated.

Finally, Python tries to assign the result back into `t[0]`.

That final assignment fails because tuples do not support item assignment.

The mutation already happened before the error was raised.


In [23]:
t = ([1, 2], [3, 4])

try:
    t[0] += [99]
except TypeError as ex:
    error_name = type(ex).__name__

assert error_name == "TypeError"
assert t == ([1, 2, 99], [3, 4])

print(error_name)
print(t)
print("All assertions passed.")


TypeError
([1, 2, 99], [3, 4])
All assertions passed.


## Problem 12 — Function Side Effects with List `+=`


Predict the output.

Does the function mutate the caller's list?


In [24]:
def add_values(seq, values):
    seq += values
    return seq


numbers = [1, 2]
result = add_values(numbers, [3, 4])

print(numbers)
print(result)
print(numbers is result)


[1, 2, 3, 4]
[1, 2, 3, 4]
True


### Solution 12


Expected output:

```text
[1, 2, 3, 4]
[1, 2, 3, 4]
True
```

Inside the function, `seq` and `numbers` refer to the same list object.

Since list `+=` mutates in place, the caller's list is changed.


In [25]:
def add_values(seq, values):
    seq += values
    return seq


numbers = [1, 2]
result = add_values(numbers, [3, 4])

assert numbers == [1, 2, 3, 4]
assert result == [1, 2, 3, 4]
assert numbers is result

print("All assertions passed.")


All assertions passed.


## Problem 13 — Function Behavior with Tuple `+=`


Predict the output.

This is the same function as before, but now the input is a tuple.


In [26]:
def add_values(seq, values):
    seq += values
    return seq


numbers = (1, 2)
result = add_values(numbers, (3, 4))

print(numbers)
print(result)
print(numbers is result)


(1, 2)
(1, 2, 3, 4)
False


### Solution 13


Expected output:

```text
(1, 2)
(1, 2, 3, 4)
False
```

Inside the function, `seq += values` creates a new tuple and rebinds the local name `seq`.

The original tuple outside the function is unchanged.


In [27]:
def add_values(seq, values):
    seq += values
    return seq


numbers = (1, 2)
result = add_values(numbers, (3, 4))

assert numbers == (1, 2)
assert result == (1, 2, 3, 4)
assert numbers is not result

print("All assertions passed.")


All assertions passed.


## Problem 14 — Safe Function Design: Avoid Mutating Caller Data


The following function is intended to return a new list, not mutate the original.

Fix it.

Buggy version:

```python
def add_items(items, extra):
    items += extra
    return items
```


In [28]:
def add_items(items, extra):
    items += extra
    return items


original = [1, 2]
new_items = add_items(original, [3, 4])

print(original)
print(new_items)
print(original is new_items)


[1, 2, 3, 4]
[1, 2, 3, 4]
True


### Solution 14


A safer version is:

```python
def add_items(items, extra):
    result = list(items)
    result.extend(extra)
    return result
```

This creates a new list first, then extends that new list.

The caller's original list remains unchanged.


In [29]:
def add_items(items, extra):
    result = list(items)
    result.extend(extra)
    return result


original = [1, 2]
new_items = add_items(original, [3, 4])

assert original == [1, 2]
assert new_items == [1, 2, 3, 4]
assert original is not new_items

print(original)
print(new_items)
print("All assertions passed.")


[1, 2]
[1, 2, 3, 4]
All assertions passed.


## Problem 15 — Mutable Default Argument Bug with `+=`


Predict the output.

This combines two common issues:

- mutable default arguments
- in-place mutation with `+=`


In [30]:
DEFAULT_TAGS = ["new"]

def create_record(name, tags=DEFAULT_TAGS):
    tags += [name.lower()]
    return {"name": name, "tags": tags}


r1 = create_record("Alice")
r2 = create_record("Bob")

print(r1)
print(r2)
print(DEFAULT_TAGS)
print(r1["tags"] is r2["tags"])


{'name': 'Alice', 'tags': ['new', 'alice', 'bob']}
{'name': 'Bob', 'tags': ['new', 'alice', 'bob']}
['new', 'alice', 'bob']
True


### Solution 15


Expected output:

```text
{'name': 'Alice', 'tags': ['new', 'alice', 'bob']}
{'name': 'Bob', 'tags': ['new', 'alice', 'bob']}
['new', 'alice', 'bob']
True
```

The default list is created once when the function is defined.

Each call without an explicit `tags` argument uses the same list.

Because `+=` mutates that list in place, all calls share the accumulated tags.


In [31]:
DEFAULT_TAGS = ["new"]

def create_record(name, tags=DEFAULT_TAGS):
    tags += [name.lower()]
    return {"name": name, "tags": tags}


r1 = create_record("Alice")
r2 = create_record("Bob")

assert r1["tags"] == ["new", "alice", "bob"]
assert r2["tags"] == ["new", "alice", "bob"]
assert DEFAULT_TAGS == ["new", "alice", "bob"]
assert r1["tags"] is r2["tags"]

print("All assertions passed.")


All assertions passed.


## Problem 16 — Fixing the Mutable Default Argument Bug


Rewrite `create_record` so each record gets an independent list of tags.

Requirements:

- If no tags are provided, use `["new"]`
- Do not mutate a caller-provided tag list
- Add `name.lower()` to the result


In [32]:
# Write your corrected function here.


### Solution 16


Correct version:

```python
def create_record(name, tags=None):
    if tags is None:
        tags = ["new"]
    else:
        tags = list(tags)

    tags += [name.lower()]
    return {"name": name, "tags": tags}
```

Using `None` avoids sharing a mutable default list.

Using `list(tags)` avoids mutating a caller-provided list.


In [33]:
def create_record(name, tags=None):
    if tags is None:
        tags = ["new"]
    else:
        tags = list(tags)

    tags += [name.lower()]
    return {"name": name, "tags": tags}


r1 = create_record("Alice")
r2 = create_record("Bob")

external_tags = ["vip"]
r3 = create_record("Cara", external_tags)

assert r1 == {"name": "Alice", "tags": ["new", "alice"]}
assert r2 == {"name": "Bob", "tags": ["new", "bob"]}
assert r3 == {"name": "Cara", "tags": ["vip", "cara"]}
assert external_tags == ["vip"]

print(r1)
print(r2)
print(r3)
print("All assertions passed.")


{'name': 'Alice', 'tags': ['new', 'alice']}
{'name': 'Bob', 'tags': ['new', 'bob']}
{'name': 'Cara', 'tags': ['vip', 'cara']}
All assertions passed.


## Problem 17 — Generator Consumption with List `+=`


Predict the output.

Remember that list `+=` consumes any iterable.


In [34]:
def numbers():
    print("start")
    yield 1
    print("middle")
    yield 2
    print("end")


lst = [0]
g = numbers()

lst += g

print(lst)
print(list(g))


start
middle
end
[0, 1, 2]
[]


### Solution 17


Expected output:

```text
start
middle
end
[0, 1, 2]
[]
```

The generator is consumed by `lst += g`.

Once consumed, the generator is exhausted.

Therefore `list(g)` is empty afterward.


In [35]:
def numbers():
    print("start")
    yield 1
    print("middle")
    yield 2
    print("end")


lst = [0]
g = numbers()

lst += g

assert lst == [0, 1, 2]
assert list(g) == []

print("All assertions passed.")


start
middle
end
All assertions passed.


## Problem 18 — Set Iteration Order with `+=`


Predict what is guaranteed and what is not guaranteed.

```python
lst = [0]
lst += {1, 2, 3}
print(lst)
```

Is the exact order guaranteed?


In [36]:
lst = [0]
lst += {1, 2, 3}
print(lst)


[0, 1, 2, 3]


### Solution 18


The guaranteed part is:

```python
lst[0] == 0
set(lst[1:]) == {1, 2, 3}
len(lst) == 4
```

The exact order of values coming from a set is not something you should rely on.

So do not write tests that require the result to be exactly `[0, 1, 2, 3]`.


In [37]:
lst = [0]
lst += {1, 2, 3}

assert lst[0] == 0
assert set(lst[1:]) == {1, 2, 3}
assert len(lst) == 4

print(lst)
print("All assertions passed.")


[0, 1, 2, 3]
All assertions passed.


## Problem 19 — Aliasing Through a Dictionary


Predict the output.

The list is stored in a dictionary and also referenced by another variable.


In [38]:
record = {"values": [1, 2]}
alias = record["values"]

record["values"] += [3, 4]

print(record)
print(alias)
print(record["values"] is alias)


{'values': [1, 2, 3, 4]}
[1, 2, 3, 4]
True


### Solution 19


Expected output:

```text
{'values': [1, 2, 3, 4]}
[1, 2, 3, 4]
True
```

The value stored under `"values"` is a list.

`record["values"] += [3, 4]` mutates that list in place.

The variable `alias` refers to the same list object.


In [39]:
record = {"values": [1, 2]}
alias = record["values"]

record["values"] += [3, 4]

assert record == {"values": [1, 2, 3, 4]}
assert alias == [1, 2, 3, 4]
assert record["values"] is alias

print("All assertions passed.")


All assertions passed.


## Problem 20 — Augmented Assignment to a Dictionary Key


Compare these two snippets:

Snippet A:

```python
record = {"values": [1, 2]}
alias = record["values"]
record["values"] += [3]
```

Snippet B:

```python
record = {"values": [1, 2]}
alias = record["values"]
record["values"] = record["values"] + [3]
```

How do they differ?


In [40]:
# Run both snippets and compare.

record = {"values": [1, 2]}
alias = record["values"]
record["values"] += [3]

print("A:", record, alias, record["values"] is alias)

record = {"values": [1, 2]}
alias = record["values"]
record["values"] = record["values"] + [3]

print("B:", record, alias, record["values"] is alias)


A: {'values': [1, 2, 3]} [1, 2, 3] True
B: {'values': [1, 2, 3]} [1, 2] False


### Solution 20


Snippet A mutates the original list.

Snippet B creates a new list and stores it back into the dictionary.

Expected output:

```text
A: {'values': [1, 2, 3]} [1, 2, 3] True
B: {'values': [1, 2, 3]} [1, 2] False
```


In [41]:
record = {"values": [1, 2]}
alias = record["values"]
record["values"] += [3]

assert record["values"] == [1, 2, 3]
assert alias == [1, 2, 3]
assert record["values"] is alias

record = {"values": [1, 2]}
alias = record["values"]
record["values"] = record["values"] + [3]

assert record["values"] == [1, 2, 3]
assert alias == [1, 2]
assert record["values"] is not alias

print("All assertions passed.")


All assertions passed.


## Problem 21 — Custom Class with `__add__` but No `__iadd__`


Predict the output.

The class supports `+`, but not true in-place `+=`.


In [42]:
class Box:
    def __init__(self, items):
        self.items = list(items)

    def __add__(self, other):
        return Box(self.items + list(other))

    def __repr__(self):
        return f"Box({self.items})"


box = Box([1, 2])
alias = box

before = id(box)
box += [3, 4]
after = id(box)

print(box)
print(alias)
print(before == after)
print(box is alias)


Box([1, 2, 3, 4])
Box([1, 2])
False
False


### Solution 21


Expected output:

```text
Box([1, 2, 3, 4])
Box([1, 2])
False
False
```

Since `Box` does not define `__iadd__`, Python falls back to `__add__`.

So:

```python
box += [3, 4]
```

behaves like:

```python
box = box + [3, 4]
```

The name `box` is rebound to a new object.


In [43]:
class Box:
    def __init__(self, items):
        self.items = list(items)

    def __add__(self, other):
        return Box(self.items + list(other))

    def __repr__(self):
        return f"Box({self.items})"


box = Box([1, 2])
alias = box

before = id(box)
box += [3, 4]
after = id(box)

assert box.items == [1, 2, 3, 4]
assert alias.items == [1, 2]
assert before != after
assert box is not alias

print("All assertions passed.")


All assertions passed.


## Problem 22 — Custom Class with Correct `__iadd__`


Implement true in-place addition for `Box`.

Requirements:

- Mutate the existing object
- Preserve aliases
- Return `self`


In [44]:
class Box:
    def __init__(self, items):
        self.items = list(items)

    def __add__(self, other):
        return Box(self.items + list(other))

    # Add __iadd__ here.

    def __repr__(self):
        return f"Box({self.items})"


### Solution 22


Correct implementation:

```python
def __iadd__(self, other):
    self.items.extend(other)
    return self
```

`__iadd__` should usually mutate `self` and return `self`.

That keeps object identity stable.


In [45]:
class Box:
    def __init__(self, items):
        self.items = list(items)

    def __add__(self, other):
        return Box(self.items + list(other))

    def __iadd__(self, other):
        self.items.extend(other)
        return self

    def __repr__(self):
        return f"Box({self.items})"


box = Box([1, 2])
alias = box

before = id(box)
box += [3, 4]
after = id(box)

assert box.items == [1, 2, 3, 4]
assert alias.items == [1, 2, 3, 4]
assert before == after
assert box is alias

print(box)
print(alias)
print("All assertions passed.")


Box([1, 2, 3, 4])
Box([1, 2, 3, 4])
All assertions passed.


## Problem 23 — Broken `__iadd__`: Mutates but Returns a New Object


Predict the output.

This class is intentionally badly designed.


In [46]:
class WeirdList:
    def __init__(self, values):
        self.values = list(values)

    def __iadd__(self, other):
        self.values.extend(other)
        return WeirdList([-1])

    def __repr__(self):
        return f"WeirdList({self.values})"


x = WeirdList([1, 2])
alias = x

x += [3]

print(x)
print(alias)
print(x is alias)


WeirdList([-1])
WeirdList([1, 2, 3])
False


### Solution 23


Expected output:

```text
WeirdList([-1])
WeirdList([1, 2, 3])
False
```

`__iadd__` mutates the original object first.

Then it returns a different object.

The augmented assignment binds `x` to the returned object.

So `alias` points to the mutated original, while `x` points to the new `WeirdList([-1])`.


In [47]:
class WeirdList:
    def __init__(self, values):
        self.values = list(values)

    def __iadd__(self, other):
        self.values.extend(other)
        return WeirdList([-1])

    def __repr__(self):
        return f"WeirdList({self.values})"


x = WeirdList([1, 2])
alias = x

x += [3]

assert x.values == [-1]
assert alias.values == [1, 2, 3]
assert x is not alias

print("All assertions passed.")


All assertions passed.


## Problem 24 — Custom `__imul__` for In-Place Repetition


Create a class `RepeatBox` that supports `*=`.

Expected behavior:

```python
box = RepeatBox([1, 2])
alias = box
box *= 3

print(box)
print(alias)
print(box is alias)
```

Expected output:

```text
RepeatBox([1, 2, 1, 2, 1, 2])
RepeatBox([1, 2, 1, 2, 1, 2])
True
```


In [48]:
# Implement RepeatBox here.


### Solution 24


A correct implementation defines `__imul__`.

The method mutates `self.values` and returns `self`.


In [49]:
class RepeatBox:
    def __init__(self, values):
        self.values = list(values)

    def __imul__(self, n):
        self.values *= n
        return self

    def __repr__(self):
        return f"RepeatBox({self.values})"


box = RepeatBox([1, 2])
alias = box

box *= 3

assert box.values == [1, 2, 1, 2, 1, 2]
assert alias.values == [1, 2, 1, 2, 1, 2]
assert box is alias

print(box)
print(alias)
print("All assertions passed.")


RepeatBox([1, 2, 1, 2, 1, 2])
RepeatBox([1, 2, 1, 2, 1, 2])
All assertions passed.


## Problem 25 — `__mul__` Without `__imul__`


Predict the output.

The class supports `*`, but does not define `__imul__`.


In [50]:
class RepeatBox:
    def __init__(self, values):
        self.values = list(values)

    def __mul__(self, n):
        return RepeatBox(self.values * n)

    def __repr__(self):
        return f"RepeatBox({self.values})"


box = RepeatBox([1, 2])
alias = box

before = id(box)
box *= 3
after = id(box)

print(box)
print(alias)
print(before == after)
print(box is alias)


RepeatBox([1, 2, 1, 2, 1, 2])
RepeatBox([1, 2])
False
False


### Solution 25


Expected output:

```text
RepeatBox([1, 2, 1, 2, 1, 2])
RepeatBox([1, 2])
False
False
```

Since `__imul__` is missing, Python falls back to `__mul__`.

That creates a new object, then rebinds the name `box`.


In [51]:
class RepeatBox:
    def __init__(self, values):
        self.values = list(values)

    def __mul__(self, n):
        return RepeatBox(self.values * n)

    def __repr__(self):
        return f"RepeatBox({self.values})"


box = RepeatBox([1, 2])
alias = box

before = id(box)
box *= 3
after = id(box)

assert box.values == [1, 2, 1, 2, 1, 2]
assert alias.values == [1, 2]
assert before != after
assert box is not alias

print("All assertions passed.")


All assertions passed.


## Problem 26 — Using `operator.iadd`


The `operator` module exposes augmented-assignment-style functions.

Predict the output.


In [52]:
import operator

lst = [1, 2]
alias = lst

result = operator.iadd(lst, [3, 4])

print(lst)
print(result)
print(alias)
print(result is lst)
print(alias is lst)


[1, 2, 3, 4]
[1, 2, 3, 4]
[1, 2, 3, 4]
True
True


### Solution 26


Expected output:

```text
[1, 2, 3, 4]
[1, 2, 3, 4]
[1, 2, 3, 4]
True
True
```

`operator.iadd(lst, [3, 4])` calls the in-place addition operation.

For lists, this mutates the list and returns the same list.


In [53]:
import operator

lst = [1, 2]
alias = lst

result = operator.iadd(lst, [3, 4])

assert lst == [1, 2, 3, 4]
assert result is lst
assert alias is lst

print("All assertions passed.")


All assertions passed.


## Problem 27 — `operator.iadd` with Tuples


Predict the output.

This time the operand is immutable.


In [54]:
import operator

t = (1, 2)
alias = t

result = operator.iadd(t, (3, 4))

print(t)
print(result)
print(alias)
print(result is t)
print(alias is t)


(1, 2)
(1, 2, 3, 4)
(1, 2)
False
True


### Solution 27


Expected output:

```text
(1, 2)
(1, 2, 3, 4)
(1, 2)
False
True
```

`operator.iadd` returns the result of in-place addition.

For tuples, there is no true in-place mutation.

A new tuple is returned.

Unlike augmented assignment syntax, `operator.iadd(t, ...)` does not rebind the name `t`.


In [55]:
import operator

t = (1, 2)
alias = t

result = operator.iadd(t, (3, 4))

assert t == (1, 2)
assert result == (1, 2, 3, 4)
assert alias is t
assert result is not t

print("All assertions passed.")


All assertions passed.


## Problem 28 — Subtle Difference: Function Call versus Augmented Assignment


Compare:

```python
operator.iadd(t, (3, 4))
```

with:

```python
t += (3, 4)
```

Why does one rebind `t` while the other does not?


In [56]:
import operator

t1 = (1, 2)
operator.iadd(t1, (3, 4))
print("t1:", t1)

t2 = (1, 2)
t2 += (3, 4)
print("t2:", t2)


t1: (1, 2)
t2: (1, 2, 3, 4)


### Solution 28


Expected output:

```text
t1: (1, 2)
t2: (1, 2, 3, 4)
```

`operator.iadd(t1, ...)` returns a value, but if you do not assign that return value, the name `t1` remains bound to the original tuple.

Augmented assignment syntax includes assignment back to the target.

So `t2 += (3, 4)` rebinds `t2` to the new tuple.


In [57]:
import operator

t1 = (1, 2)
operator.iadd(t1, (3, 4))

t2 = (1, 2)
t2 += (3, 4)

assert t1 == (1, 2)
assert t2 == (1, 2, 3, 4)

print("All assertions passed.")


All assertions passed.


## Problem 29 — Production Debugging: Shared Task Lists


The developer expected three independent task lists.

What is actually printed? Then fix the code.


In [58]:
tasks = [[]] * 3

for task_list in tasks:
    task_list += ["review"]

print(tasks)


[['review', 'review', 'review'], ['review', 'review', 'review'], ['review', 'review', 'review']]


### Solution 29


Actual output:

```text
[['review', 'review', 'review'], ['review', 'review', 'review'], ['review', 'review', 'review']]
```

`[[]] * 3` repeats references to the same inner list.

The loop mutates the same inner list three times.

Correct version:

```python
tasks = [[] for _ in range(3)]

for task_list in tasks:
    task_list += ["review"]
```


In [59]:
tasks = [[] for _ in range(3)]

for task_list in tasks:
    task_list += ["review"]

assert tasks == [["review"], ["review"], ["review"]]
assert tasks[0] is not tasks[1]
assert tasks[1] is not tasks[2]

print(tasks)
print("All assertions passed.")


[['review'], ['review'], ['review']]
All assertions passed.


## Problem 30 — Design Challenge: Non-Mutating Repetition


Write a function `repeat_new(seq, n)` that returns a repeated sequence without mutating the original input.

Requirements:

- If `seq` is a list, return a new list.
- If `seq` is a tuple, return a new tuple.
- Do not use `*=`.


In [60]:
# Implement repeat_new here.


### Solution 30


One straightforward implementation is:

```python
def repeat_new(seq, n):
    return seq * n
```

For both lists and tuples, `seq * n` creates a new container.

The important part is that we do not do `seq *= n`, which might mutate a list.


In [61]:
def repeat_new(seq, n):
    return seq * n


lst = [1, 2]
lst_alias = lst
lst_result = repeat_new(lst, 3)

tpl = (1, 2)
tpl_alias = tpl
tpl_result = repeat_new(tpl, 3)

assert lst == [1, 2]
assert lst_alias is lst
assert lst_result == [1, 2, 1, 2, 1, 2]
assert lst_result is not lst

assert tpl == (1, 2)
assert tpl_alias is tpl
assert tpl_result == (1, 2, 1, 2, 1, 2)
assert tpl_result is not tpl

print("All assertions passed.")


All assertions passed.


## Problem 31 — Design Challenge: Mutating Only When Safe


Write a function `extend_in_place(target, values)`.

Requirements:

- It should mutate `target` if `target` has an `extend` method.
- It should return the mutated object.
- If `target` does not support `extend`, raise `TypeError`.


In [62]:
# Implement extend_in_place here.


### Solution 31


A reasonable implementation is:

```python
def extend_in_place(target, values):
    if not hasattr(target, "extend"):
        raise TypeError("target must support extend")
    target.extend(values)
    return target
```

This is explicit. It avoids the more subtle fallback behavior of `+=`.


In [63]:
def extend_in_place(target, values):
    if not hasattr(target, "extend"):
        raise TypeError("target must support extend")
    target.extend(values)
    return target


lst = [1, 2]
alias = lst

result = extend_in_place(lst, [3, 4])

assert lst == [1, 2, 3, 4]
assert result is lst
assert alias is lst

try:
    extend_in_place((1, 2), (3, 4))
except TypeError as ex:
    assert "extend" in str(ex)
else:
    raise AssertionError("Expected TypeError")

print("All assertions passed.")


All assertions passed.


## Problem 32 — Advanced Debugging: Unexpected Mutation in a Cache


A cache stores a list of events.

The developer wants `get_events_with_marker` to return a new list, leaving the cache unchanged.

Find the bug and fix it.


In [64]:
cache = {
    "events": ["login", "view"]
}

def get_events_with_marker(marker):
    events = cache["events"]
    events += [marker]
    return events


result = get_events_with_marker("debug")

print("cache:", cache)
print("result:", result)
print(cache["events"] is result)


cache: {'events': ['login', 'view', 'debug']}
result: ['login', 'view', 'debug']
True


### Solution 32


Bug:

```python
events = cache["events"]
events += [marker]
```

`events` refers to the same list stored inside the cache.

The `+=` operation mutates that cached list.

Correct version:

```python
def get_events_with_marker(marker):
    events = cache["events"] + [marker]
    return events
```

or:

```python
def get_events_with_marker(marker):
    events = list(cache["events"])
    events.append(marker)
    return events
```


In [65]:
cache = {
    "events": ["login", "view"]
}

def get_events_with_marker(marker):
    events = list(cache["events"])
    events.append(marker)
    return events


result = get_events_with_marker("debug")

assert cache == {"events": ["login", "view"]}
assert result == ["login", "view", "debug"]
assert cache["events"] is not result

print("cache:", cache)
print("result:", result)
print("All assertions passed.")


cache: {'events': ['login', 'view']}
result: ['login', 'view', 'debug']
All assertions passed.


## Problem 33 — Advanced Prediction: Mixed Mutability


Predict the output.

This example contains a tuple, a list inside the tuple, aliases, and `*=`.


In [66]:
inner = [1]
container = (inner,)
alias = inner

inner *= 3
container[0] += [2]

print(inner)
print(alias)
print(container)
print(container[0] is inner)


TypeError: 'tuple' object does not support item assignment

### Solution 33


This code raises a `TypeError` at:

```python
container[0] += [2]
```

But before the error is raised, the list inside the tuple is mutated.

A complete demonstration should catch the error:

```python
inner = [1]
container = (inner,)
alias = inner

inner *= 3

try:
    container[0] += [2]
except TypeError:
    print("TypeError")

print(inner)
print(alias)
print(container)
print(container[0] is inner)
```

Expected output:

```text
TypeError
[1, 1, 1, 2]
[1, 1, 1, 2]
([1, 1, 1, 2],)
True
```


In [67]:
inner = [1]
container = (inner,)
alias = inner

inner *= 3

try:
    container[0] += [2]
except TypeError:
    got_type_error = True
else:
    got_type_error = False

assert got_type_error
assert inner == [1, 1, 1, 2]
assert alias == [1, 1, 1, 2]
assert container == ([1, 1, 1, 2],)
assert container[0] is inner

print("All assertions passed.")


All assertions passed.


## Problem 34 — Refactoring Challenge: Make Mutation Obvious


The following code works, but the mutation is not obvious enough.

Refactor it to make the mutation explicit.

Original:

```python
def add_to_bucket(buckets, key, values):
    buckets[key] += values
```

Assume `buckets[key]` is always a list.


In [68]:
def add_to_bucket(buckets, key, values):
    buckets[key] += values


### Solution 34


A clearer version is:

```python
def add_to_bucket(buckets, key, values):
    buckets[key].extend(values)
```

This communicates that the existing list is intentionally mutated.

`+=` is not wrong here, but `extend` is more explicit.


In [69]:
def add_to_bucket(buckets, key, values):
    buckets[key].extend(values)


buckets = {"a": [1, 2]}
alias = buckets["a"]

add_to_bucket(buckets, "a", [3, 4])

assert buckets == {"a": [1, 2, 3, 4]}
assert alias == [1, 2, 3, 4]
assert buckets["a"] is alias

print("All assertions passed.")


All assertions passed.


## Problem 35 — Refactoring Challenge: Avoid Mutation


The following code mutates the input list.

Refactor it to return a new list instead.

Original:

```python
def with_footer(lines):
    lines += ["-- end --"]
    return lines
```


In [70]:
def with_footer(lines):
    lines += ["-- end --"]
    return lines


### Solution 35


A safer non-mutating version is:

```python
def with_footer(lines):
    return list(lines) + ["-- end --"]
```

This returns a new list and leaves the caller's object unchanged.


In [71]:
def with_footer(lines):
    return list(lines) + ["-- end --"]


lines = ["alpha", "beta"]
alias = lines

result = with_footer(lines)

assert lines == ["alpha", "beta"]
assert alias is lines
assert result == ["alpha", "beta", "-- end --"]
assert result is not lines

print("All assertions passed.")


All assertions passed.


## Problem 36 — Final Challenge: Classify Each Operation


For each operation below, classify it as either:

- mutates existing object
- creates new object and rebinds name
- mutates first, then raises an error

Operations:

```python
a = [1, 2]
a += [3]

b = [1, 2]
b = b + [3]

t = (1, 2)
t += (3,)

s = "a"
s += "b"

x = [[0]]
x *= 3

y = ([1],)
y[0] += [2]
```


In [72]:
# Use this cell for experiments if needed.


### Solution 36


Classification:

```python
a = [1, 2]
a += [3]
```

Mutates existing object.

```python
b = [1, 2]
b = b + [3]
```

Creates new object and rebinds name.

```python
t = (1, 2)
t += (3,)
```

Creates new object and rebinds name.

```python
s = "a"
s += "b"
```

Creates new object and rebinds name.

```python
x = [[0]]
x *= 3
```

Mutates existing outer list, but repeats references to the same inner list.

```python
y = ([1],)
y[0] += [2]
```

Mutates the inner list first, then raises `TypeError` when Python attempts to assign back into the tuple.


In [73]:
# Verification snippets

a = [1, 2]
alias_a = a
a += [3]
assert a is alias_a
assert a == [1, 2, 3]

b = [1, 2]
alias_b = b
b = b + [3]
assert b is not alias_b
assert alias_b == [1, 2]
assert b == [1, 2, 3]

t = (1, 2)
alias_t = t
t += (3,)
assert t is not alias_t
assert alias_t == (1, 2)
assert t == (1, 2, 3)

s = "a"
alias_s = s
s += "b"
assert s == "ab"
assert alias_s == "a"

x = [[0]]
alias_x = x
x *= 3
assert x is alias_x
assert x[0] is x[1] is x[2]

y = ([1],)
try:
    y[0] += [2]
except TypeError:
    pass
else:
    raise AssertionError("Expected TypeError")

assert y == ([1, 2],)

print("All assertions passed.")


All assertions passed.


# Summary

Use these rules as a practical checklist:

1. `list += iterable` mutates the list in place.
2. `list *= n` mutates the list in place.
3. `tuple += tuple` creates a new tuple and rebinds the name.
4. `str += str` creates a new string and rebinds the name.
5. `list + other_list` creates a new list.
6. `append(x)` adds `x` as one element.
7. `extend(iterable)` adds each item from the iterable.
8. Repetition of nested mutable objects repeats references, not deep copies.
9. Augmented assignment to an item, such as `obj[index] += value`, may mutate first and then fail during assignment.
10. In custom classes, define `__iadd__` and `__imul__` carefully.
11. A well-behaved mutable `__iadd__` or `__imul__` usually mutates `self` and returns `self`.
12. Prefer explicit methods like `extend` when you want mutation to be obvious.
13. Prefer copying first when you want to avoid mutating caller-owned data.
